# Fig 4C / 4D / S9 — Meta-analysis of divergent populations, AFR + EUR (data processing)

Simulated meta-analysis mixing AFR and EUR samples at known proportions.

**Outputs:**
- `figure_data/Fig4C.AFR_EUR_Weight_Estimation.rds` — estimated vs. true AFR proportion (+ `_accuracy.csv`)
- `figure_data/Fig4D.AFR_EUR_Mix_Power_FDR.rds` — power / FDR of mixed vs. single LD (bootstrap 95% CI)
- `figure_data/FigS9D.AFR_EUR_Hetero_Weight_Estimation.rds` — proportion estimation when EUR and AFR have distinct causal variants (+ `_accuracy.csv`)

In [1]:
source("../../R/utils.R")

res_file   <- "/hpc/users/sunh17/haochen/rss_qc/2026_output/NEW_EUR_AFR_fine/summary/result_susie_mix_summary_summary.rds"
n_GWAS     <- 800

In [2]:
res <- readRDS(res_file)

## Fig 4C: weight estimation

In [3]:
res <- readRDS(res_file) %>%
  mutate(weight = as.numeric(str_extract(input_file, "(?<=prop_)[0-9.]+")))

om <- res %>%
  filter(method == "mix_refpanel", !is.na(omega_1)) %>%
  transmute(h2g, causal, weight = as.numeric(weight), omega_1)

om_acc <- om %>%
  group_by(h2g, causal, weight) %>%
  summarize(mean_omega = mean(omega_1), sd_omega = sd(omega_1),
            MAE = mean(abs(omega_1 - weight)), n = n(), .groups = "drop")
write_csv(om_acc, file.path(data_dir, "Fig4C.AFR_EUR_Weight_Estimation_accuracy.csv"))

save_data(om, "Fig4C.AFR_EUR_Weight_Estimation")

## Fig 4D: power / FDR
Note: the SuSiEx rows (`susiex_refpanel`) are read in but are not in `method_labels`, so the `method %in% method_order` filter drops them (as in the original notebook).

In [4]:
method_labels <- c(
  gen_refpanel    = "Oracle",
  eur_refpanel    = "EUR only",
  afr_refpanel    = "AFR only",
  manual_refpanel = "SuSiE-rRSS"
)
method_order <- names(pal_meta_afr_eur)       # SuSiE-rRSS, Oracle, AFR only, EUR only

res <- readRDS(res_file) %>%
  mutate(weight = as.numeric(str_extract(input_file, "(?<=prop_)[0-9.]+")))
susie_x_res <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_susiex/summary/NEW_EUR_AFR_susiex_susiex_mix_summary_summary.rds") %>%
  filter(method == "susiex_refpanel") %>% mutate(weight = as.numeric(prop))
res <- bind_rows(res, susie_x_res)

cell_reps <- res %>%
  { if ("fit_ok" %in% names(.)) filter(., fit_ok) else . } %>%
  mutate(weight = as.numeric(weight),
         method = recode(method, !!!method_labels)) %>%
  filter(!round(weight, 3) %in% c(0.10, 0.90),
         method %in% method_order)

## bootstrap 95% CIs (boot_cell_mix() in R/utils.R)
n_boot <- 1000
set.seed(1)
summary_ci <- cell_reps %>%
  group_by(method, weight, h2g, causal) %>%
  group_modify(~ boot_cell_mix(.x, B = n_boot)) %>%
  ungroup()

In [5]:
## ---- plot-ready: long Power / FDR, NCP and causal facets ---------------------
plot_df <- summary_ci %>%
  mutate(ncp_num = n_GWAS * parse_h2g(h2g),
         ncp = fct_reorder(paste0("NCP = ", round(ncp_num)), ncp_num))   # whole numbers (formatC "g" gave 1.2e+02)

w_levels <- sort(unique(plot_df$weight))
pl <- plot_df %>%
  transmute(method, weight, ncp, causal,
            Power_est = power, Power_lo = power_lo, Power_hi = power_hi,
            FDR_est   = FDR,   FDR_lo   = FDR_lo,   FDR_hi   = FDR_hi) %>%
  pivot_longer(-c(method, weight, ncp, causal),
               names_to = c("metric", ".value"), names_sep = "_") %>%
  mutate(
    metric = factor(metric, levels = c("Power", "FDR")),
    causal = factor(paste0(causal, " causal"),
                    levels = paste0(sort(unique(as.integer(causal))), " causal")),
    weight = factor(weight, levels = w_levels, labels = paste0(round(w_levels * 100), "%")),
    method = factor(method, levels = method_order)
  )
stopifnot(!any(is.na(pl$method)))

save_data(pl, "Fig4D.AFR_EUR_Mix_Power_FDR")

## Fig S9: heterogeneous causal variants across populations

In [6]:
library(parallel)

root  <- "/hpc/users/sunh17/haochen/rss_qc/2026_output/NEW_EUR_AFR_Hetero_Fine/result/"
files <- list.files(root, pattern = "\\.rds$", full.names = TRUE, recursive = TRUE)

read_one <- function(f) {
  tryCatch({
    res <- readRDS(f)
    est <- res$est_afr_prop
    tibble(
      sample       = as.integer(str_extract(basename(f), "(?<=sample_)[0-9]+")),
      ncausal      = as.integer(str_extract(f, "(?<=/causal_)[0-9]+")),
      h2g          = as.numeric(str_extract(f, "(?<=/h2g_)[0-9.]+")),
      true_prop    = as.numeric(str_extract(f, "(?<=/prop_)[0-9.]+")),
      est_afr_prop = if (length(est)) as.numeric(est[1]) else NA_real_,
      file         = basename(f)
    )
  }, error = function(e) { message("FAILED: ", f, " : ", conditionMessage(e)); NULL })
}

tb <- mclapply(files, read_one, mc.cores = 4) %>%
  bind_rows() %>%
  arrange(h2g, ncausal, true_prop, sample)

tb                       # per-file table: meta-params + est_afr_prop

sample,ncausal,h2g,true_prop,est_afr_prop,file
<int>,<int>,<dbl>,<dbl>,<dbl>,<chr>
0,1,0.025,0,0,sample_0_ncausal_1_h2g_0.025_prop_0.0_mix_pop.rds
1,1,0.025,0,0,sample_1_ncausal_1_h2g_0.025_prop_0.0_mix_pop.rds
2,1,0.025,0,0,sample_2_ncausal_1_h2g_0.025_prop_0.0_mix_pop.rds
3,1,0.025,0,0,sample_3_ncausal_1_h2g_0.025_prop_0.0_mix_pop.rds
4,1,0.025,0,0,sample_4_ncausal_1_h2g_0.025_prop_0.0_mix_pop.rds
5,1,0.025,0,0,sample_5_ncausal_1_h2g_0.025_prop_0.0_mix_pop.rds
6,1,0.025,0,0,sample_6_ncausal_1_h2g_0.025_prop_0.0_mix_pop.rds
7,1,0.025,0,0,sample_7_ncausal_1_h2g_0.025_prop_0.0_mix_pop.rds
9,1,0.025,0,0,sample_9_ncausal_1_h2g_0.025_prop_0.0_mix_pop.rds


In [7]:
om <- tb %>%
  transmute(h2g, causal = ncausal, weight = true_prop, omega_1 = est_afr_prop) %>%
  filter(!is.na(omega_1))

om_acc <- om %>%
  group_by(h2g, causal, weight) %>%
  summarize(mean_omega = mean(omega_1), sd_omega = sd(omega_1),
            MAE = mean(abs(omega_1 - weight)), n = n(), .groups = "drop")
write_csv(om_acc, file.path(data_dir, "FigS9D.AFR_EUR_Hetero_Weight_Estimation_accuracy.csv"))

save_data(om, "FigS9D.AFR_EUR_Hetero_Weight_Estimation")